# 統計模型與損失函數

對應 DLI：01、02。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/05_statistical_models_loss.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/05-statistical-models-loss.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

[背景導讀：從統計到 AI 的模型設計](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/statistics-to-ai.md)

## 學習目標與先備知識

先讀條件機率與標準化。需要理解函數是「輸入數值得到輸出」；本課補充指數與對數，不要求先學最佳化推導。



## 補充｜ 從一條直線開始預測延遲

假設你要預測一套影像辨識服務的處理時間。輸入 $x$ 是排在前面的請求數，輸出 $y$ 是量到的延遲（毫秒）。本章的資料、數字與練習都是教學示例。

把一條直線當作第一個模型：

$$\hat y=b+wx$$

$\hat y$ 是模型的預測，$b$ 是截距，代表 $x=0$ 時模型的預測。$w$ 是斜率，描述模型預測裡，每多一個請求，延遲改變幾毫秒。這個斜率來自資料的擬合，若要解讀為因果效果，還需要合適的實驗或辨識條件。

### 殘差是圖上哪一段？

在相同的 $x_i$，從直線上的預測值走到實際資料點，垂直距離帶正負號就是殘差 $r_i=y_i-\hat y_i$。正值代表低估，負值代表高估。這是用已觀測的答案與已擬合模型算出的差，理論模型中的隨機誤差則是相對於未知的真實條件平均。

| 前方請求數 $x$ | 實測 $y$（ms） | 起始直線 $\hat y=2x$ | 殘差 $r$（ms） | 殘差平方（ms²） |
|---:|---:|---:|---:|---:|
| 1 | 2 | 2 | 0 | 0 |
| 2 | 3 | 4 | −1 | 1 |
| 3 | 7 | 6 | 1 | 1 |

這三個殘差的總和是 0，但只有第一筆預測準確。若直接加總，低估和高估會抵銷。平方後再平均，得到 $MSE=(0+1+1)/3\approx0.667$ ms²。MSE 把大偏差加重。例如 4 ms 的偏差是 2 ms 的兩倍，平方懲罰卻是四倍。

### 迴歸怎麼選出直線？

普通最小平方法（OLS）選擇 $b,w$，使 $\sum_i(y_i-b-wx_i)^2$ 最小。同一批資料上，除以固定的筆數只會改變數值尺度，因此使 SSE 最小的直線也會使 MSE 最小。這個例子的最小平方直線是 $\hat y=-1+2.5x$，預測為 $[1.5,4,6.5]$，殘差為 $[0.5,-1,0.5]$，MSE 為 0.5 ms²。截距為負也提醒我們，模型在觀察範圍外的預測需要另外驗證。

**進一步手算。** $\bar x=2$，$\bar y=4$。斜率是 $\sum_i(x_i-\bar x)(y_i-\bar y)/\sum_i(x_i-\bar x)^2=5/2$，截距是 $\bar y-w\bar x=4-5=-1$。對新資料能否預測得好，要看留出的資料與殘差圖。若殘差在某個區間總是偏正，先檢查模型是否漏掉轉折或重要變數，再考慮改模型。[來源：NIST 線性最小平方迴歸](https://www.itl.nist.gov/div898/handbook/pmd/section1/pmd141.htm)

**自我檢核。** 若預測 9 ms，實測 12 ms，殘差為 $+3$ ms，代表低估。重新擬合後，各點的殘差都可能改變。接著可看 [第 06 課的 ReLU 轉折](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md) 與 [第 10 課的兩種殘差比較](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/10-text-statistics.md)。

## 1. 從猜一個數字到猜一個類別

**來源摘要 [S5](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)：**線性模型結合特徵與係數；邏輯斯迴歸用於分類，可以產生類別機率。名稱中雖有「迴歸」，用途仍可為分類。

本教材自編例子：用影像亮度 x 預測某一類手勢。先算分數 $z=b+wx$，w 決定輸入變化的影響，b 是截距。若直接把 z 當機率，可能得到負數或超過 1；因此二元分類可用 sigmoid：

$$p=\frac{1}{1+e^{-z}}$$

z=0 時 p=0.5；z 很大時 p 接近 1，z 很負時 p 接近 0。e 約為 2.718，是自然指數的底數。



## 2. 真實答案、預測與誤差

| 記號 | 意思 |
|---|---|
| $x_i$ | 第 i 筆輸入特徵 |
| $y_i$ | 真實答案 |
| $\hat{y}_i$ | 預測值或預測類別，需交代定義 |
| $p_i$ | 第 i 筆屬於正類的預測機率 |
| $L$ | 損失，衡量預測和答案的不一致 |

若預測的是連續數值，可用平均平方誤差：

$$MSE=\frac1n\sum_i(y_i-\hat y_i)^2$$

**自編手算：**真實延遲 [10,20]，預測 [12,16]，誤差 [−2,4]，平方 [4,16]，平均為 10 ms²。損失是 10，不是「準確率 10%」。



## 3. 為什麼分類會用交叉熵？

**來源摘要 [S6](https://scikit-learn.org/stable/modules/model_evaluation.html)：**log loss／交叉熵使用機率預測。二元分類中，單筆損失可寫成 $-[y\ln p+(1-y)\ln(1-p)]$；多類別則關注真實類別被分配的機率。

先看答案為正類 y=1 時，公式變為 −ln p。ln 是自然對數：ln(1)=0，當 p 越接近 0，−ln p 越大。

本教材自編對照：

| 對真實類別給的機率 | 損失 $-\ln p$ | 直覺 |
|---:|---:|---|
| 0.9 | 約 0.105 | 給對答案高分 |
| 0.5 | 約 0.693 | 不確定 |
| 0.1 | 約 2.303 | 給對答案低分 |

不要把機率 0 或 1 直接帶入未保護的 log 程式，因為 ln(0) 不有限。套件通常以數值保護處理，神經網路也常直接使用 logits 的穩定損失函數。



### 補充｜先定義要預測的答案，再選輸出和損失

對同一套辨識服務，預測「延遲幾毫秒」與判斷「延遲是否超過 12 ms」是兩個不同的任務。前者的標籤是 5、7、9、14、19 ms，後者把同一批觀測改標成 0、0、0、1、1。**標籤的意義決定輸出的格式，損失則把任務在意的錯誤換成可最佳化的數值。**這個決定可以與隱藏層的啟動函數分開考慮。

| 任務與標籤 | 輸出如何解讀 | 常用損失與理由 |
|---|---|---|
| 預測實數延遲，例如 14 ms | 一個實數，迴歸常從恆等輸出開始，有硬性範圍時可另外設計 | MSE 加重大偏差，等變異數的獨立高斯誤差假設可導出最小平方。MAE 對偏差的懲罰是線性的，適合把條件中位數當目標。Huber 在小誤差下採平方，大誤差下近似線性，可依有意義的誤差尺度設定轉折點 |
| 是否超時，標籤為 0 或 1 | 一個 logit $z$，推論時以 sigmoid 解讀為正類分數 | BCE 對應二元結果的負對數概似。PyTorch 用 BCEWithLogitsLoss 直接接 logit |
| 一筆請求只屬於一種狀態 | $K$ 個 logits，softmax 後的機率總和為 1 | CrossEntropyLoss 直接接 logits。一般類別索引格式每筆只給一個整數類別 |
| 可能同時超時又發生辨識錯誤 | 每個標籤有一個 logit，各自轉成 sigmoid 分數 | BCEWithLogitsLoss 逐標籤計算。各分數不必加總為 1 |
| 給定前文，預測下一個 token | 每個位置輸出一組詞彙表 logits | token 級交叉熵，通常在需要監督的位置取平均 |

**為什麼用負對數？** 以二元超時為例，觀測到 1 時模型給答案的機率是 $p$，觀測到 0 時則是 $1-p$。在樣本給定輸入且條件獨立的建模假設下，整批標籤的概似是每筆機率相乘。取負對數後，乘積變成可逐筆相加的損失，較可能出現的答案帶來較小的損失。若實際超時，一個模型給 $p=0.8$，損失約 0.223，另一個給 $p=0.2$，損失約 1.609。這兩個數值可用程式核對。

PyTorch 的 BCEWithLogitsLoss 已包含 sigmoid 的對應計算，CrossEntropyLoss 也直接處理 logits。訓練時把原始分數傳給這兩個 API，推論需要閱讀機率時才另外轉換。這是 API 的輸入契約。選擇警報閾值時，還要用驗證集檢查漏報與誤報的成本。若服務有少數真實的高延遲事件，應把那些事件的表現另外列出，避免單看一個平均損失。可接著閱讀 [機率與分類](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/02-probability-classification.md) 與 [分類模型評估](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/07-classification-evaluation.md)。

[來源：PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.BCEWithLogitsLoss.html) · [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.CrossEntropyLoss.html) · [Transformer 原論文 §3.4](https://arxiv.org/html/1706.03762v7#S3.SS4)

## 4. 相同 Accuracy，損失可能差很多

本教材程式的真實類別是 [0,1]；兩組正類機率分別為 [0.6,0.6] 與 [0.99,0.99]。以 0.5 為閾值，兩者都預測 [1,1]，所以 Accuracy 都是 50%。

第一組平均交叉熵約 0.714；第二組約 2.308。第二組對第一筆錯誤更自信，受到較大懲罰。請把手算結果對照 log_loss 輸出。



## 5. 模型怎麼學？怎麼連到 DLI？

模型學習是調整參數使訓練損失下降。梯度可先理解成「損失對某參數變動的局部敏感度」；以學習率控制每次調整幅度。下降的訓練損失不保證未見資料表現提升，仍要看驗證集。

下方使用 scikit-learn 的 8×8 digits 建立輕量基準，**不是 MNIST**。連到 DLI 時，請比較輸入形狀、輸出類別數與損失函數，避免把本例的分數當作 DLI 模型結果。

### 補充｜模型怎麼從損失知道每個權重該往哪裡改？

想像用 $\hat y=b+wx$ 預測延遲。若 $x=2$、$w=3$、$b=1$，模型預測 7 ms，實測 9 ms。殘差 $r=y-\hat y=2$ ms，單筆平方損失 $L=r^2=4$ ms²。

前向運算從 $x$ 算到 $\hat y$，再從 $\hat y$ 和答案算出 $L$。反向傳播（backpropagation）沿這條計算路徑反向套用鏈鎖律，算出損失對各參數的偏導數。此例中，$w$ 每變動一點，預測的變動率為 $x=2$。損失對預測的變動率為 $-2r=-4$。相乘得到：

$$\frac{\partial L}{\partial w}
=\frac{\partial L}{\partial \hat y}\frac{\partial\hat y}{\partial w}
=(-4)(2)=-8.$$

負梯度表示在目前這個點，小幅增加 $w$ 會使損失下降。若暫時固定 $b$，用梯度下降與學習率 0.1，權重更新為 $3-0.1(-8)=3.8$。新預測為 8.6 ms，單筆損失為約 0.16 ms²。**反向傳播負責算斜率，最佳化器根據斜率及其更新規則調整參數。** 在 PyTorch 中，常見的兩步是先呼叫 loss.backward()，再呼叫 optimizer.step()。

多層模型也遵循同一個原理。每向前回到一層，就再乘上該層的局部導數。下一課會說明多次相乘時，梯度為什麼可能變得很小。[第 06 課：Sigmoid 與梯度消失](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md) · [PyTorch 自動微分教學](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

## 自我檢核

1. y=0，正類機率 p=0.9 時，應代入 −ln(0.9) 還是 −ln(0.1)？
2. Accuracy 不變時，交叉熵為何仍可能改善？
3. 訓練損失降低是否足以證明模型更適合部署？

### 參考解答

1. −ln(0.1)，因真實類別是負類。
2. 分類是否跨過閾值沒有改變，但給真實類別的機率可能更合適。
3. 否。還需驗證泛化、資料代表性及部署需求。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

### 迴歸手算驗證

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

queue = np.array([1., 2., 3.])
observed = np.array([2., 3., 7.])
design = np.column_stack([np.ones(queue.size), queue])
intercept, slope = np.linalg.lstsq(design, observed, rcond=None)[0]
prediction = intercept + slope * queue
residual = observed - prediction

print("Initial line MSE:", np.mean((observed - 2 * queue) ** 2))
print("OLS intercept and slope:", np.round([intercept, slope], 3))
print("Residuals and MSE:", np.round(residual, 3), np.mean(residual**2))
np.testing.assert_allclose(residual, [0.5, -1, 0.5])

fig, ax = plt.subplots()
ax.scatter(queue, observed, label="Observed")
ax.plot(queue, prediction, label="Fitted line")
ax.vlines(queue, np.minimum(observed, prediction),
          np.maximum(observed, prediction), linestyles="dashed", label="Residuals")
ax.set(xlabel="Requests ahead", ylabel="Latency (ms)")
ax.legend()
plt.show()

### 用 logits 驗證二元交叉熵

In [ ]:
import numpy as np

labels = np.array([0., 1.])
logits = np.log(np.array([4., 4.]))  # sigmoid(log(4)) = 0.8
probabilities = 1 / (1 + np.exp(-logits))
# Stable binary cross-entropy from logits: softplus(z) - y*z.
loss_per_example = np.logaddexp(0, logits) - labels * logits
print("Positive-class probabilities:", probabilities)
print("Binary targets and losses:", list(zip(labels.tolist(), loss_per_example.round(3).tolist())))
np.testing.assert_allclose(loss_per_example, -np.log([0.2, 0.8]))

In [ ]:
import numpy as np
from sklearn.metrics import log_loss, accuracy_score
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
truth = np.array([0, 1])
for name, prob in [('moderate', [0.6, 0.6]), ('high confidence', [0.99, 0.99])]:
    prob = np.array(prob)
    print(name, 'accuracy=', accuracy_score(truth, prob >= .5),
          'cross entropy=', log_loss(truth, np.column_stack([1-prob, prob])))
X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)
print('Digits baseline accuracy:', model.score(X_test, y_test))

## 如何解讀

scikit-learn digits 是 8×8 的資料，並非 DLI 使用的 MNIST；此處用它避免下載與 GPU 需求。這是基準示範，測試集不能反覆用於挑設定。

## 練習

把一個預測機率從 0.6 改成 0.99，觀察正確與錯誤時交叉熵的不同。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S5**：[scikit-learn，Logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression)。
- **S6**：[scikit-learn，Metrics and scoring（Log loss、Precision、Recall）](https://scikit-learn.org/stable/modules/model_evaluation.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
